# 02 Empirical Data Audit

## Purpose

Audit a canonical long-format empirical panel or a synthetic fallback. This notebook is descriptive only and should not be used for substantive claims.

## Data / config used

- local path: `data/raw/macro_stress_example.csv` if present
- synthetic fallback otherwise
- output directory: `outputs/notebooks/02_empirical_data_audit/`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from dynsys_econometrics.data import load_time_series_csv
from dynsys_econometrics.contracts import TimeSeriesFrame

output_dir = Path("outputs/notebooks/02_empirical_data_audit")
output_dir.mkdir(parents=True, exist_ok=True)

input_path = Path("data/raw/macro_stress_example.csv")
if input_path.exists():
    panel = load_time_series_csv(input_path, series_id="macro_stress")
else:
    dates = pd.date_range("2000-01-31", periods=180, freq="ME")
    values = np.cumsum(np.random.default_rng(17).normal(scale=0.2, size=len(dates)))
    panel = pd.DataFrame({"date": dates, "series_id": "macro_stress", "value": values})

validated = TimeSeriesFrame(panel)


In [ ]:
frame = validated.to_frame()
audit = pd.DataFrame(
    {
        "n_rows": [len(frame)],
        "n_series": [frame["series_id"].nunique()],
        "start_date": [frame["date"].min()],
        "end_date": [frame["date"].max()],
        "n_missing_values": [int(frame["value"].isna().sum())],
    }
)
audit.to_csv(output_dir / "empirical_audit.csv", index=False)
frame.head().to_csv(output_dir / "panel_head.csv", index=False)
audit


## Limitations

This audit notebook is descriptive. The diagnostics used here do not identify causal mechanisms, and fallback synthetic data must not be treated as empirical evidence.
